# MNIST Digit Classifier — Train → INT8 Quantize → FPGA Export

Simple fully-connected network: **784 → 150 (ReLU) → 10**, trained with backprop (Adam) in PyTorch.

Pipeline in this notebook:
1. Load MNIST, build & train the FC net (target: >85% test accuracy — this simple net will actually land around 97-98%).
2. Evaluate float32 baseline accuracy.
3. Calibrate activation ranges, then quantize weights + activations to **INT8** (symmetric, per-tensor), biases to INT32.
4. Run a **pure-integer** forward pass (the same arithmetic an FPGA int8 MAC array would do) and measure quantized accuracy on the full test set.
5. Export weights/biases/scales to `.bin` and `.mem` (Verilog `$readmemh`) files for FPGA loading.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 1. Data

Plain `ToTensor()` scaling gives pixel values in `[0, 1]` (non-negative, no mean/std normalization). This keeps the numeric range simple and predictable, which makes the later quantization step easier to reason about.

In [ ]:
transform = transforms.ToTensor()  # scales pixels to [0, 1], shape (1, 28, 28)

train_set = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_set  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=256, shuffle=False)

print(f"Train samples: {len(train_set)}, Test samples: {len(test_set)}")


## 2. Model — 784 → 150 (ReLU) → 10

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, in_dim=784, hidden=150, out_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        x = x.view(x.size(0), -1)      # flatten 28*28 -> 784
        x = self.relu(self.fc1(x))
        x = self.fc2(x)                # raw logits (CrossEntropyLoss applies softmax internally)
        return x

model = SimpleNN().to(device)
print(model)


## 3. Train (backprop via Adam)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 12

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_set)
    test_acc = evaluate(model, test_loader)
    print(f"Epoch {epoch:2d}/{EPOCHS}  loss={train_loss:.4f}  test_acc={test_acc*100:.2f}%")


In [ ]:
float_acc = evaluate(model, test_loader)
print(f"Final FLOAT32 test accuracy: {float_acc*100:.2f}%")
assert float_acc >= 0.85, "Should easily clear 85% with this architecture — check training if not."


## 4. INT8 Quantization

We quantize so the eventual FPGA design only ever does **integer** multiply-accumulates:

- **Weights**: per-tensor symmetric INT8. `scale_w = max(|W|) / 127`, `W_q = round(W / scale_w)` clipped to `[-127, 127]`.
- **Activations** (input image, hidden layer output): also symmetric INT8, but the scale is **calibrated once** from data (not recomputed per batch), since real hardware needs a fixed, baked-in scale.
- **Biases**: INT32, scaled by `scale_x * scale_w` of the layer that consumes them (standard practice, e.g. TFLite-style int8 quantization) so they can be added directly to the int32 accumulator.

Forward pass per layer, all integer math:
```
acc_int32 = x_int8 @ W_int8.T + b_int32
out_float = acc_int32 * (scale_x * scale_w)     # dequantize just to apply ReLU / requantize
```
Note: since dequantizing is just a positive scalar multiply, it doesn't change `argmax` ordering — on real FPGA hardware you can skip the final dequant on the last layer and argmax the raw int32 accumulator directly.


In [ ]:
def quantize_symmetric(x, num_bits=8):
    """Symmetric per-tensor quantization. Returns (int8_array, scale)."""
    max_abs = np.max(np.abs(x))
    qmax = 2 ** (num_bits - 1) - 1  # 127
    scale = max_abs / qmax if max_abs > 0 else 1.0
    q = np.clip(np.round(x / scale), -qmax - 1, qmax).astype(np.int8)
    return q, scale

# --- pull trained float weights out of the model ---
W1 = model.fc1.weight.detach().cpu().numpy()   # (150, 784)
b1 = model.fc1.bias.detach().cpu().numpy()     # (150,)
W2 = model.fc2.weight.detach().cpu().numpy()   # (10, 150)
b2 = model.fc2.bias.detach().cpu().numpy()     # (10,)

W1_q, W1_scale = quantize_symmetric(W1)
W2_q, W2_scale = quantize_symmetric(W2)
print("W1 scale:", W1_scale, " W2 scale:", W2_scale)


In [ ]:
# --- calibrate activation scales on the training set (post-training static calibration) ---
def collect_activation_stats(model, loader, n_batches=50):
    model.eval()
    max_input, max_hidden = 0.0, 0.0
    with torch.no_grad():
        for i, (x, _) in enumerate(loader):
            if i >= n_batches:
                break
            x = x.to(device)
            flat = x.view(x.size(0), -1)
            max_input = max(max_input, flat.abs().max().item())
            hidden = torch.relu(model.fc1(flat))
            max_hidden = max(max_hidden, hidden.abs().max().item())
    return max_input, max_hidden

max_input, max_hidden = collect_activation_stats(model, train_loader)
qmax = 127
input_scale  = max_input / qmax
hidden_scale = max_hidden / qmax
print(f"input_scale={input_scale:.6f}  hidden_scale={hidden_scale:.6f}")

# biases quantized to int32 using the scale of the accumulator they land in
bias1_scale = input_scale * W1_scale
bias2_scale = hidden_scale * W2_scale
b1_q = np.round(b1 / bias1_scale).astype(np.int32)
b2_q = np.round(b2 / bias2_scale).astype(np.int32)


## 5. Pure-integer quantized inference + accuracy check

In [ ]:
def quantize_activation(x_float, scale, num_bits=8):
    qmax = 2 ** (num_bits - 1) - 1
    return np.clip(np.round(x_float / scale), -qmax - 1, qmax).astype(np.int8)

def quantized_forward(x_uint8_img):
    """x_uint8_img: float numpy array in [0,1], shape (N, 784). Returns predicted class indices."""
    x_q = quantize_activation(x_uint8_img, input_scale)                     # int8

    # Layer 1: int32 accumulator
    acc1 = x_q.astype(np.int32) @ W1_q.T.astype(np.int32) + b1_q            # (N, 150) int32
    hidden_float = acc1 * bias1_scale                                       # dequantize to apply ReLU
    hidden_float = np.maximum(hidden_float, 0.0)                            # ReLU
    hidden_q = quantize_activation(hidden_float, hidden_scale)              # requantize to int8

    # Layer 2: int32 accumulator (logits) — argmax is scale-invariant, no need to dequantize
    acc2 = hidden_q.astype(np.int32) @ W2_q.T.astype(np.int32) + b2_q       # (N, 10) int32
    return np.argmax(acc2, axis=1)

# run over the full test set
all_preds, all_labels = [], []
for x, y in test_loader:
    x_flat = x.view(x.size(0), -1).numpy()
    preds = quantized_forward(x_flat)
    all_preds.append(preds)
    all_labels.append(y.numpy())

all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)
quant_acc = (all_preds == all_labels).mean()

print(f"FLOAT32 test accuracy:  {float_acc*100:.2f}%")
print(f"INT8    test accuracy:  {quant_acc*100:.2f}%")
print(f"Accuracy drop:          {(float_acc - quant_acc)*100:.2f} pts")


## 6. Export weights for FPGA loading

Exports, per layer:
- `*_weights_int8.bin` — raw int8 weight matrix, row-major
- `*_bias_int32.bin` — raw int32 bias vector
- `*_weights.mem` / `*_bias.mem` — hex text files, one value per line, for Verilog `$readmemh`
- `quant_params.json` — shapes + all scales needed to interpret the fixed-point values at inference time


In [ ]:
import os, json

out_dir = "fpga_export"
os.makedirs(out_dir, exist_ok=True)

def write_mem_file(arr, path, bits=8):
    """Write one hex value per line (twos-complement bit pattern), for $readmemh."""
    fmt = "{:02x}\n" if bits == 8 else "{:08x}\n"
    utype = np.uint8 if bits == 8 else np.uint32
    with open(path, "w") as f:
        for v in arr.flatten():
            f.write(fmt.format(int(utype(v))))

def export_layer(name, w_q, b_q):
    w_q.tofile(os.path.join(out_dir, f"{name}_weights_int8.bin"))
    b_q.tofile(os.path.join(out_dir, f"{name}_bias_int32.bin"))
    write_mem_file(w_q, os.path.join(out_dir, f"{name}_weights.mem"), bits=8)
    write_mem_file(b_q, os.path.join(out_dir, f"{name}_bias.mem"), bits=32)

export_layer("fc1", W1_q, b1_q)
export_layer("fc2", W2_q, b2_q)

quant_params = {
    "input_scale": input_scale,
    "hidden_scale": hidden_scale,
    "fc1": {"weight_shape": list(W1_q.shape), "weight_scale": W1_scale,
            "bias_shape": list(b1_q.shape), "bias_scale": bias1_scale},
    "fc2": {"weight_shape": list(W2_q.shape), "weight_scale": W2_scale,
            "bias_shape": list(b2_q.shape), "bias_scale": bias2_scale},
    "notes": "Weights/activations: int8 symmetric per-tensor. Biases: int32, scale = act_scale * weight_scale. "
             "Layer1 out = ReLU(x_int8 @ W1_int8.T + b1_int32), dequant by bias1_scale before ReLU, "
             "then requantize by hidden_scale. Layer2 out (logits) = hidden_int8 @ W2_int8.T + b2_int32; "
             "argmax directly on the int32 accumulator."
}
with open(os.path.join(out_dir, "quant_params.json"), "w") as f:
    json.dump(quant_params, f, indent=2)

print("Exported files:")
for fn in sorted(os.listdir(out_dir)):
    print(" -", fn)


### Notes for the actual FPGA implementation

- These scales are **not** powers of two, so `out_float = acc * scale` isn't a simple bit-shift. If your FPGA design needs shift-only rescaling, snap each scale to the nearest power of two before quantizing weights (small extra accuracy cost) — happy to adjust the notebook to do that if you want it.
- `W1_q` is stored row-major as `(150, 784)` — i.e. `hidden_idx * 784 + input_idx`. `W2_q` is `(10, 150)` — `class_idx * 150 + hidden_idx`. Adjust the `.mem` read order in your Verilog testbench to match, or transpose before export if your MAC array expects column-major.
- The `.bin` files are the same data as the `.mem` files, just as raw binary instead of hex text — use whichever your toolchain / testbench prefers.
